In [1]:
# 07_adversarial.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 07 - Adversarial debiasing (in-processing) and a controlled signal test.
#
# Adversarial debiasing via gradient reversal, implemented from scratch in NumPy (no
# deep-learning framework dependency, for full reproducibility): a shared encoder feeds
# a target predictor and an adversary that tries to recover the sensitive attribute;
# reversing the adversary's gradient drives the representation toward sensitive-
# attribute invariance. Because this is a neural-network family, a neural baseline
# (lambda = 0) is trained for a within-family comparison; results are kept separate
# from the XGBoost-based techniques in notebooks 04-05.
#
# Reviewer-driven points addressed here:
#   - The "explainability artifact" diagnosis is applied to EVERY dataset, not only
#     the weak-signal one: for each run we record whether a rise in RCFS coincides
#     with a FALL in c_perf and/or c_fair, which signals that the gain sits on the
#     explainability axis (c_expl rises as the predictor sheds effective features).
#   - A CONTROLLED signal-degradation experiment on a single dataset (Taiwan, held at
#     a fixed sample size and feature set) injects label noise in the TRAIN split at
#     increasing rates, so the effect of signal strength on technique ordering and on
#     the adversarial outcome is isolated from differences in size/features/attributes.
# Fairness uses the favourable (non-default) outcome, worst-case across sensitive
# attributes; the decision threshold is validation-tuned. No paths are printed.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
np.random.seed(42)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
EXPL_KSTAR = 10
GAMMA_DEFAULT = 0.5
W_FAIR, W_EXPL = 1.0, 1.0
EPOCHS = 80
LR = 2e-3
HIDDEN = 64
ADV_LAMBDA = 1.0
LAMBDA_GRID = [0.0, 0.1, 0.5, 1.0, 2.0]
SHAP_EVAL = 200
SHAP_BG = 50
SHAP_NSAMPLES = 100
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  random_state=RANDOM_STATE, n_jobs=-1)
NOISE_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.4]
DEGRADE_KEY = "taiwan"          # strongest-signal dataset for the controlled test
DEGRADE_N = 6000                # fixed train size across noise levels

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)
DATASETS = [("German Credit", "german"), ("Taiwan CC Default", "taiwan"),
            ("Credit Card Approval", "credit_card_approval")]
PRIVILEGED = {"sex": "male", "age_group": "old", "education": ["graduate", "university"]}


def load_split(key, part):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_{part}.parquet"))


def get_Xy(df, key):
    meta = META[key]
    drop = [meta["target_col"]] + meta["sensitive_cols"]
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    return X, df[meta["target_col"]].astype(int)


def priv_mask_for(series, privileged):
    if isinstance(privileged, (list, tuple)):
        return series.isin(list(privileged)).values
    return (series == privileged).values


def c_fair_worstcase(df_sens, sens_cols, fav, y_pred):
    best = None; y_pred = np.asarray(y_pred)
    for a in sens_cols:
        s = df_sens[a]; pm = priv_mask_for(s, PRIVILEGED[a]); um = (~pm) & s.notna().values
        fm = (y_pred == fav)
        r_p = fm[pm].mean() if pm.sum() else np.nan
        r_u = fm[um].mean() if um.sum() else np.nan
        if not (np.isfinite(r_p) and np.isfinite(r_u)) or r_p == 0:
            continue
        di, spd = r_u / r_p, r_u - r_p
        sc = 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))
        best = sc if best is None else min(best, sc)
    return best


def c_expl_from_shap(sv, k_star=EXPL_KSTAR):
    mean_abs = np.abs(sv).mean(axis=0); total = mean_abs.sum()
    if total == 0:
        return 0.0
    p = mean_abs / total; p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    return float(min(1.0, k_star / np.exp(-np.sum(p * np.log(p)))))


def harmonic(cs, ws):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    return np.nan if not pairs else sum(w for _, w in pairs) / sum(w / c for c, w in pairs)


def rcfs(c_perf, comp, gamma):
    return np.nan if not np.isfinite(comp) else (1 - gamma) * c_perf + gamma * comp


def _relu(z):
    return np.maximum(0.0, z)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


class AdvNet:
    # Encoder (two hidden ReLU layers) feeding a predictor head and an adversary head.
    # Trained full-batch with Adam; the adversary gradient is reversed (scaled by -lam)
    # before it reaches the encoder, implementing domain-adversarial debiasing.
    def __init__(self, in_dim, hidden=HIDDEN, seed=RANDOM_STATE):
        rng = np.random.default_rng(seed)
        he = lambda a, b: rng.standard_normal((a, b)) * np.sqrt(2.0 / a)
        self.W1, self.b1 = he(in_dim, hidden), np.zeros(hidden)
        self.W2, self.b2 = he(hidden, hidden), np.zeros(hidden)
        self.Wp, self.bp = he(hidden, 1) * 0.1, np.zeros(1)
        self.Wa, self.ba = he(hidden, 1) * 0.1, np.zeros(1)
        self._keys = ["W1", "b1", "W2", "b2", "Wp", "bp", "Wa", "ba"]
        self.m = {k: np.zeros_like(getattr(self, k)) for k in self._keys}
        self.v = {k: np.zeros_like(getattr(self, k)) for k in self._keys}
        self.t = 0

    def _enc(self, X):
        z1 = X @ self.W1 + self.b1; h1 = _relu(z1)
        z2 = h1 @ self.W2 + self.b2; h2 = _relu(z2)
        return z1, h1, z2, h2

    def predict_proba1(self, X):
        _, _, _, h2 = self._enc(X)
        return _sigmoid((h2 @ self.Wp + self.bp).ravel())

    def fit(self, X, y, s, lam, pos_w, epochs=EPOCHS, lr=LR):
        y = y.astype(float); s = s.astype(float); N = len(y)
        ws = np.where(y == 1, pos_w, 1.0)
        for _ in range(epochs):
            z1, h1, z2, h2 = self._enc(X)
            p = _sigmoid((h2 @ self.Wp + self.bp).ravel())
            dyl = (ws * (p - y) / N).reshape(-1, 1)
            dWp = h2.T @ dyl; dbp = dyl.sum(0); dh2 = dyl @ self.Wp.T
            if lam > 0:
                ps = _sigmoid((h2 @ self.Wa + self.ba).ravel())
                dsl = ((ps - s) / N).reshape(-1, 1)
                dWa = h2.T @ dsl; dba = dsl.sum(0)
                dh2 = dh2 + (-lam) * (dsl @ self.Wa.T)   # gradient reversal into encoder
            else:
                dWa = np.zeros_like(self.Wa); dba = np.zeros_like(self.ba)
            dz2 = dh2 * (z2 > 0); dW2 = h1.T @ dz2; db2 = dz2.sum(0); dh1 = dz2 @ self.W2.T
            dz1 = dh1 * (z1 > 0); dW1 = X.T @ dz1; db1 = dz1.sum(0)
            grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2,
                     "Wp": dWp, "bp": dbp, "Wa": dWa, "ba": dba}
            self.t += 1; b1c = 1 - 0.9 ** self.t; b2c = 1 - 0.999 ** self.t
            for k, g in grads.items():
                self.m[k] = 0.9 * self.m[k] + 0.1 * g
                self.v[k] = 0.999 * self.v[k] + 0.001 * (g * g)
                setattr(self, k, getattr(self, k)
                        - lr * (self.m[k] / b1c) / (np.sqrt(self.v[k] / b2c) + 1e-8))
        return self


def train_net(X, y, s, in_dim, lam, pos_w, epochs=EPOCHS, lr=LR, seed=RANDOM_STATE):
    return AdvNet(in_dim, seed=seed).fit(X.astype(np.float64), y, s, lam, pos_w, epochs, lr)


def proba_net(net, X):
    return net.predict_proba1(X.astype(np.float64))


def shap_net(net, X_bg, X_ev, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    bg = X_bg[rng.choice(len(X_bg), size=min(SHAP_BG, len(X_bg)), replace=False)]
    ev = X_ev[rng.choice(len(X_ev), size=min(SHAP_EVAL, len(X_ev)), replace=False)]
    sv = shap.KernelExplainer(lambda d: net.predict_proba1(d.astype(np.float64)), bg
                              ).shap_values(ev, nsamples=SHAP_NSAMPLES, silent=True)
    return np.array(sv)


def val_threshold(proba_val, rate):
    return np.quantile(proba_val, 1 - rate)


def score_net(net, test_sens, sens_cols, fav, X_tr, X_val, X_te, y_te, rate):
    proba_val = proba_net(net, X_val); proba = proba_net(net, X_te)
    y_pred = (proba >= val_threshold(proba_val, rate)).astype(int)
    auc = roc_auc_score(y_te.values, proba)
    c_perf = max(0.0, (auc - 0.5) / 0.5)
    c_fair = c_fair_worstcase(test_sens, sens_cols, fav, y_pred)
    c_expl = c_expl_from_shap(shap_net(net, X_tr, X_te))
    comp = harmonic([c_fair, c_expl], [W_FAIR, W_EXPL])
    return dict(AUC=auc, c_perf=c_perf, c_fair=c_fair, c_expl=c_expl,
                Compliance=comp, RCFS=rcfs(c_perf, comp, GAMMA_DEFAULT))

# --- Part 1 & 2: main comparison and lambda sweep across the three datasets ---
rows, sweep_rows = [], []
for display_name, key in DATASETS:
    meta = META[key]; sens_cols = meta["sensitive_cols"]; prim = sens_cols[0]; fav = meta["favourable_label"]
    tr, val, te = load_split(key, "train"), load_split(key, "val"), load_split(key, "test")
    X_tr_df, y_tr = get_Xy(tr, key); X_val_df, _ = get_Xy(val, key); X_te_df, y_te = get_Xy(te, key)
    X_val_df = X_val_df[X_tr_df.columns]; X_te_df = X_te_df[X_tr_df.columns]
    X_tr = X_tr_df.values.astype(np.float32); X_val = X_val_df.values.astype(np.float32)
    X_te = X_te_df.values.astype(np.float32); in_dim = X_tr.shape[1]
    s_tr = (tr[prim].values == PRIVILEGED[prim]).astype(np.float32)
    rate = float(y_tr.mean())
    n_pos = int(y_tr.sum()); n_neg = int((y_tr == 0).sum()); pos_w = (n_neg / n_pos) if n_pos else 1.0
    base_m = None
    for lam in LAMBDA_GRID:
        net = train_net(X_tr, y_tr.values.astype(np.float32), s_tr, in_dim, lam, pos_w)
        m = score_net(net, te[sens_cols], sens_cols, fav, X_tr, X_val, X_te, y_te, rate)
        m.update({"dataset": display_name, "adv_lambda": lam})
        sweep_rows.append(m)
        if lam == 0.0:
            base_m = m
        if lam in (0.0, ADV_LAMBDA):
            r = dict(m); r["technique"] = "nn_baseline" if lam == 0.0 else "adversarial"
            rows.append(r)
        print(f"[{display_name:>22}] lambda={lam:<4} AUC={m['AUC']:.3f} "
              f"c_fair={m['c_fair']:.3f} c_expl={m['c_expl']:.3f} RCFS={m['RCFS']:.3f}")

adv_df = pd.DataFrame(rows)[["dataset", "technique", "AUC", "c_perf", "c_fair",
                            "c_expl", "Compliance", "RCFS"]].round(4)
adv_df.to_csv(os.path.join(TAB_DIR, "07_adversarial_rcfs.csv"), index=False)
print(adv_df.to_string(index=False))

comp = (adv_df.pivot(index="dataset", columns="technique", values="RCFS")
        .reindex(["German Credit", "Taiwan CC Default", "Credit Card Approval"]))
comp["adversarial_beats_baseline"] = comp["adversarial"] > comp["nn_baseline"]
comp.to_csv(os.path.join(TAB_DIR, "07_adversarial_comparison.csv"))
print(comp.to_string())

adv_sweep = pd.DataFrame(sweep_rows)[["dataset", "adv_lambda", "AUC", "c_perf",
                                      "c_fair", "c_expl", "Compliance", "RCFS"]].round(4)
adv_sweep.to_csv(os.path.join(TAB_DIR, "07_adversarial_lambda_sweep.csv"), index=False)

# --- Part 3: explainability-artifact diagnosis, applied to all datasets ---
diag = []
for ds in ["German Credit", "Taiwan CC Default", "Credit Card Approval"]:
    sub = adv_sweep[adv_sweep["dataset"] == ds].set_index("adv_lambda")
    base = sub.loc[0.0]
    best_lam = sub.drop(index=0.0)["RCFS"].idxmax()
    b = sub.loc[best_lam]
    rcfs_up = b["RCFS"] > base["RCFS"]
    perf_down = b["c_perf"] < base["c_perf"]
    fair_down = b["c_fair"] < base["c_fair"]
    expl_up = b["c_expl"] > base["c_expl"]
    diag.append({"dataset": ds, "best_lambda>0": best_lam,
                 "dRCFS": round(b["RCFS"] - base["RCFS"], 4),
                 "dC_perf": round(b["c_perf"] - base["c_perf"], 4),
                 "dC_fair": round(b["c_fair"] - base["c_fair"], 4),
                 "dC_expl": round(b["c_expl"] - base["c_expl"], 4),
                 "artifact_flag": bool(rcfs_up and (perf_down or fair_down) and expl_up)})
diag_df = pd.DataFrame(diag)
diag_df.to_csv(os.path.join(TAB_DIR, "07_adversarial_lambda_summary.csv"), index=False)
print("Adversarial diagnosis (artifact_flag = RCFS up while performance/fairness fall "
      "and explainability rises):")
print(diag_df.to_string(index=False))

# --- Part 4: controlled signal degradation on a single dataset (Taiwan) ---
meta = META[DEGRADE_KEY]; sens_cols = meta["sensitive_cols"]; prim = sens_cols[0]; fav = meta["favourable_label"]
tr = load_split(DEGRADE_KEY, "train"); val = load_split(DEGRADE_KEY, "val"); te = load_split(DEGRADE_KEY, "test")
X_tr_df, y_tr = get_Xy(tr, DEGRADE_KEY); X_val_df, _ = get_Xy(val, DEGRADE_KEY); X_te_df, y_te = get_Xy(te, DEGRADE_KEY)
X_val_df = X_val_df[X_tr_df.columns]; X_te_df = X_te_df[X_tr_df.columns]
rng = np.random.default_rng(RANDOM_STATE)
# Fixed-size train subsample so size/features are held constant across noise levels.
sub_idx = rng.choice(len(X_tr_df), size=min(DEGRADE_N, len(X_tr_df)), replace=False)
X_tr_sub = X_tr_df.iloc[sub_idx].reset_index(drop=True)
y_tr_sub = y_tr.iloc[sub_idx].reset_index(drop=True)
s_tr_sub = tr[prim].iloc[sub_idx].reset_index(drop=True)
rate = float(y_tr_sub.mean())


def reweighing_weights(y, s):
    w = np.ones(len(y), float); n = len(y)
    for yy in np.unique(y):
        for ss in pd.unique(s):
            m = (y == yy) & (s.values == ss)
            if m.sum():
                w[m] = ((y == yy).mean() * (s.values == ss).mean()) / (m.sum() / n)
    return w


def reject_thresholds(pv, s_val, r):
    return {g: (np.quantile(pv[s_val.values == g], 1 - r) if (s_val.values == g).any() else 0.5)
            for g in pd.unique(s_val)}


def tree_c_expl(model, X):
    try:
        sv = shap.TreeExplainer(model).shap_values(X if len(X) <= 1000 else X.sample(1000, random_state=RANDOM_STATE))
    except Exception:
        return np.nan
    if isinstance(sv, list):
        sv = sv[-1]
    return c_expl_from_shap(sv)

deg_rows = []
spw = ((y_tr_sub == 0).sum() / max(1, (y_tr_sub == 1).sum()))
s_val = val[prim]
for noise in NOISE_LEVELS:
    y_noisy = np.array(y_tr_sub.values, dtype=int, copy=True)
    flip = rng.random(len(y_noisy)) < noise
    y_noisy[flip] = 1 - y_noisy[flip]
    y_noisy = pd.Series(y_noisy)
    rate_n = float(y_noisy.mean())
    tech_rcfs = {}
    for tech in ["baseline", "reweighing", "reject_option"]:
        m = XGBClassifier(scale_pos_weight=((y_noisy == 0).sum() / max(1, (y_noisy == 1).sum())), **XGB_PARAMS)
        if tech == "reweighing":
            m.fit(X_tr_sub, y_noisy, sample_weight=reweighing_weights(y_noisy.values, s_tr_sub))
        else:
            m.fit(X_tr_sub, y_noisy)
        pv = m.predict_proba(X_val_df)[:, 1]; pt = m.predict_proba(X_te_df)[:, 1]
        if tech == "reject_option":
            tm = reject_thresholds(pv, s_val, rate_n)
            yh = np.array([int(p >= tm.get(g, 0.5)) for p, g in zip(pt, te[prim].values)])
        else:
            yh = (pt >= np.quantile(pv, 1 - rate_n)).astype(int)
        auc = roc_auc_score(y_te.values, pt)
        cp = max(0.0, (auc - 0.5) / 0.5)
        cf = c_fair_worstcase(te[sens_cols], sens_cols, fav, yh)
        ce = tree_c_expl(m, X_te_df)
        tech_rcfs[tech] = dict(AUC=auc, RCFS=rcfs(cp, harmonic([cf, ce], [W_FAIR, W_EXPL]), GAMMA_DEFAULT))
    # Adversarial vs NN baseline at the same noise level.
    Xtr = X_tr_sub.values.astype(np.float32); Xval = X_val_df.values.astype(np.float32)
    Xte = X_te_df.values.astype(np.float32)
    s_bin = (s_tr_sub.values == PRIVILEGED[prim]).astype(np.float32)
    nn_scores = {}
    for lam, name in [(0.0, "nn_baseline"), (ADV_LAMBDA, "adversarial")]:
        net = train_net(Xtr, y_noisy.values.astype(np.float32), s_bin, Xtr.shape[1], lam, spw)
        nn_scores[name] = score_net(net, te[sens_cols], sens_cols, fav, Xtr, Xval, Xte, y_te, rate_n)
    best_tech = max(tech_rcfs, key=lambda t: tech_rcfs[t]["RCFS"])
    deg_rows.append({
        "noise": noise, "AUC_baseline": round(tech_rcfs["baseline"]["AUC"], 4),
        "best_xgb_technique": best_tech,
        "RCFS_baseline": round(tech_rcfs["baseline"]["RCFS"], 4),
        "RCFS_reweighing": round(tech_rcfs["reweighing"]["RCFS"], 4),
        "RCFS_reject_option": round(tech_rcfs["reject_option"]["RCFS"], 4),
        "nn_baseline_RCFS": round(nn_scores["nn_baseline"]["RCFS"], 4),
        "adversarial_RCFS": round(nn_scores["adversarial"]["RCFS"], 4),
        "adv_beats_nn": bool(nn_scores["adversarial"]["RCFS"] > nn_scores["nn_baseline"]["RCFS"]),
        "adv_c_perf": round(nn_scores["adversarial"]["c_perf"], 4),
        "adv_c_fair": round(nn_scores["adversarial"]["c_fair"], 4),
        "adv_c_expl": round(nn_scores["adversarial"]["c_expl"], 4)})
    print(f"[Taiwan degrade] noise={noise} AUC={tech_rcfs['baseline']['AUC']:.3f} "
          f"best_xgb={best_tech} adv_beats_nn={deg_rows[-1]['adv_beats_nn']}")
deg_df = pd.DataFrame(deg_rows)
deg_df.to_csv(os.path.join(TAB_DIR, "07_signal_degradation.csv"), index=False)
print("Part 4 - controlled signal degradation (Taiwan, fixed size/features):")
print(deg_df.to_string(index=False))
print("Notebook 07 complete.")


/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[         German Credit] lambda=0.0  AUC=0.748 c_fair=0.790 c_expl=0.311 RCFS=0.472


[         German Credit] lambda=0.1  AUC=0.757 c_fair=0.817 c_expl=0.326 RCFS=0.490


[         German Credit] lambda=0.5  AUC=0.770 c_fair=0.892 c_expl=0.334 RCFS=0.513


[         German Credit] lambda=1.0  AUC=0.777 c_fair=0.906 c_expl=0.335 RCFS=0.522


...

[Taiwan degrade] noise=0.0 AUC=0.748 best_xgb=reweighing adv_beats_nn=True


[Taiwan degrade] noise=0.1 AUC=0.719 best_xgb=reweighing adv_beats_nn=True


[Taiwan degrade] noise=0.2 AUC=0.715 best_xgb=baseline adv_beats_nn=True


[Taiwan degrade] noise=0.3 AUC=0.673 best_xgb=baseline adv_beats_nn=False


[Taiwan degrade] noise=0.4 AUC=0.619 best_xgb=baseline adv_beats_nn=False
Part 4 - controlled signal degradation (Taiwan, fixed size/features):
 noise  AUC_baseline best_xgb_technique  RCFS_baseline  RCFS_reweighing  RCFS_reject_option  nn_baseline_RCFS  adversarial_RCFS  adv_beats_nn  adv_c_perf  adv_c_fair  adv_c_expl
   0.0        0.7476         reweighing         0.6268           0.6284              0.6243            0.6998            0.7111          True      0.5002      0.9683      0.8800
   0.1        0.7189         reweighing         0.5813           0.5851              0.5825            0.6553            0.6724          True      0.4800      0.9560      0.7894
   0.2        0.7150           baseline         0.5642           0.5636              0.5632            0.6479            0.6632          True      0.4730      0.9070      0.8059
   0.3        0.6735           baseline         0.5118           0.5086              0.5117            0.6177            0.6055         False   